# Notebook 14 — Causal pruning validation (digit MLP · CIFAR CNN · ImageNet CNN)

Extends the notebook-13 pruning validation from the even/odd MLP to the three remaining pruning-capable models, in this order: the **digit MLP** (`SimpleMLP` 784→40→20→10, MNIST), the **CIFAR CNN** (`SmallCNN`, CIFAR-10), and the **ImageNet CNN** (pretrained SqueezeNet 1.1, 8 super-categories). The question per model: does zeroing the weights BFT ranks **most important** for a class circuit hurt *that class* more than the bystander classes, and more than the standard baselines?

Per (seed, target class) observation it runs `ablation_sweep` over a fraction grid with six rankings — `bft_top`, `bft_bottom`, `magnitude`, `act_magnitude`, `taylor`, `random` — evaluated as per-class accuracy on the full held-out test set. Both CNNs use the same **weight-level** pruning as the MLPs (conv kernels flattened to `(C_out, C_in·kH·kW)`), replacing the coarser filter-level proxy the retired nb09 ablation section used; for SqueezeNet the prunable pool is exactly the traced squeeze spine, for **all** rankings, so the comparison stays fair. Observations: 5 seeds × 10 classes (MLP), up to 5 seeds × 10 classes (CNN, skipping missing checkpoints), 1 pretrained model × 8 categories (ImageNet).

Follows the notebook-13 cluster conventions: one execution = the whole experiment, `NB14_MODE` (`local` | `cluster`) picks the compute profile, and every seed (ImageNet: every category) **checkpoints** to `data/results/nb14_pruning_<exp>.json` — one JSON per model, same schema as nb13's, so an interrupted run is still usable and the paper figure can be drawn from the JSONs alone. See the final cell for how to run on the cluster.

## §0 · Setup, compute profile & shared helpers

In [ ]:
import os, sys, json, warnings
sys.path.insert(0, '..')

import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader, Subset
from scipy.stats import wilcoxon, ttest_rel

from src import (load_experiment, get_transform, get_loaders_from_config,
                 collect_layer_dicts, bft, ablation_sweep)

warnings.filterwarnings('ignore')
REPO = os.path.abspath('..')                     # notebook lives in notebooks/
MODE = os.environ.get('NB14_MODE', 'local')
DEVICE = torch.device('cuda' if torch.cuda.is_available() else
                      ('mps' if torch.backends.mps.is_available() else 'cpu'))

FIG_DIR    = os.path.join(REPO, 'figs', '14_pruning_digit_cnn_imagenet')
RES_DIR    = os.path.join(REPO, 'data', 'results')
MODEL_ROOT = os.path.join(REPO, 'data', 'models')
for d in (FIG_DIR, RES_DIR):
    os.makedirs(d, exist_ok=True)

# Compute profile. 'local' is a laptop smoke test — numbers are NOT publication-grade.
if MODE == 'cluster':
    N_TRACE      = None      # trace on the full (confidence-filtered) sample set
    BFT_MAX_ITER = None      # -> bft default (500), matching notebooks 02/03/05
    SEED_SUBSET  = range(5)  # MLP + CNN; ImageNet has a single pretrained model
    CLASS_CAP    = None      # every class / category becomes a pruning target
    EVAL_CAP     = None      # per-class accuracy on the full test set
    # nb13's grid + 0.005/0.01 at the low end: the local smoke test showed the
    # 32k-weight digit MLP's target class already saturating to 0 by f=0.05, so
    # the region where circuit specificity resolves sits below nb13's 0.02.
    FRACTIONS    = [0.005, 0.01, 0.02, 0.05, 0.10, 0.15, 0.20, 0.30, 0.40, 0.50]
else:
    N_TRACE      = 600
    BFT_MAX_ITER = 120
    SEED_SUBSET  = range(1)
    CLASS_CAP    = 2
    EVAL_CAP     = 1500
    FRACTIONS    = [0.05, 0.20, 0.40]

FRAC_STAT   = 0.20               # fraction the significance tests are run at
ABL_METHODS = ['bft_top', 'bft_bottom', 'magnitude', 'act_magnitude', 'taylor', 'random']


def target_classes(n_classes):
    cs = list(range(n_classes))
    return cs if CLASS_CAP is None else cs[:CLASS_CAP]


def n_rand(exp_cfg):
    return exp_cfg['n_random_repeats'] if MODE == 'cluster' else 2


def capped_loader(ds, batch_size=256):
    """Evaluation loader; EVAL_CAP-limited in local mode (classes stay mixed)."""
    if EVAL_CAP is not None and len(ds) > EVAL_CAP:
        ds = Subset(ds, list(range(EVAL_CAP)))
    return DataLoader(ds, batch_size=batch_size, shuffle=False)


def _bft_iter():
    return {} if BFT_MAX_ITER is None else {'max_iter': BFT_MAX_ITER}


def jsonable(o):
    if isinstance(o, dict):
        return {str(k): jsonable(v) for k, v in o.items()}
    if isinstance(o, (list, tuple)):
        return [jsonable(v) for v in o]
    if isinstance(o, np.ndarray):
        return o.tolist()
    if isinstance(o, (np.floating, np.integer)):
        return float(o)
    return o if isinstance(o, (float, int, str, bool)) or o is None else str(o)


class Recorder:
    """Per-experiment results dict with atomic checkpointing (nb13 convention)."""

    def __init__(self, exp):
        self.exp = exp
        self.path = os.path.join(RES_DIR, f'nb14_pruning_{exp}.json')
        self.results = {'experiment': f'{exp}_pruning', 'mode': MODE}
        self.completed = []

    def checkpoint(self, section=None):
        if section and section not in self.completed:
            self.completed.append(section)
        self.results['completed_sections'] = list(self.completed)
        tmp = self.path + '.tmp'
        with open(tmp, 'w') as f:
            json.dump(jsonable(self.results), f, indent=1)
        os.replace(tmp, self.path)
        print(f'  [checkpoint] {self.exp}:{section or ""} -> '
              f'{os.path.relpath(self.path, REPO)} ({len(self.completed)} sections)')


def obs_curves(o, method, n_classes):
    """(target_accs, bystander_accs) over [0]+FRACTIONS for one observation."""
    d = o['target_class']
    others = [c for c in range(n_classes) if c != d]
    t = [o['baseline'][d]] + [o['curves'][method][f][d] for f in FRACTIONS]
    b = [np.mean([o['baseline'][c] for c in others])] + \
        [np.mean([o['curves'][method][f][c] for c in others]) for f in FRACTIONS]
    return np.array(t), np.array(b)


def aggregate_and_test(per_obs, n_classes):
    """nb13 §3 for an arbitrary class count.

    Per method: mean ± sd across observations of target / bystander accuracy at
    each fraction (fraction-0 baseline prepended). Tests, paired over the
    observations: bft_top target vs bystander drop at FRAC_STAT (specificity,
    Wilcoxon) and bft_top vs each baseline ranking on target-drop AUC (t-test).
    """
    agg = {'fractions': [0.0] + list(FRACTIONS), 'n_obs': len(per_obs), 'methods': {}}
    for m in ABL_METHODS:
        T = np.stack([obs_curves(o, m, n_classes)[0] for o in per_obs])
        B = np.stack([obs_curves(o, m, n_classes)[1] for o in per_obs])
        agg['methods'][m] = {
            'target_mean': T.mean(0), 'target_sd': T.std(0),
            'bystander_mean': B.mean(0), 'bystander_sd': B.std(0)}

    drops = {m: {'target': [], 'bystander': [], 'auc': []} for m in ABL_METHODS}
    for o in per_obs:
        d = o['target_class']
        base_t = o['baseline'][d]
        others = [c for c in range(n_classes) if c != d]
        for m in ABL_METHODS:
            drops[m]['target'].append(base_t - o['curves'][m][FRAC_STAT][d])
            drops[m]['bystander'].append(np.mean(
                [o['baseline'][c] - o['curves'][m][FRAC_STAT][c] for c in others]))
            drops[m]['auc'].append(np.mean(
                [base_t - o['curves'][m][f][d] for f in FRACTIONS]))

    stats = {'frac_stat': FRAC_STAT, 'n_obs': len(per_obs), 'drops': drops, 'tests': {}}
    t_bt = np.array(drops['bft_top']['target'])
    b_bt = np.array(drops['bft_top']['bystander'])
    if len(t_bt) >= 2 and np.any(t_bt != b_bt):
        w, p = wilcoxon(t_bt, b_bt)
        stats['tests']['bft_top_target_vs_bystander'] = {'wilcoxon_stat': float(w),
                                                         'p': float(p)}
    for m in ABL_METHODS:
        if m == 'bft_top' or len(per_obs) < 2:
            continue
        t, p = ttest_rel(drops['bft_top']['auc'], drops[m]['auc'])
        stats['tests'][f'bft_top_vs_{m}_target_auc'] = {'t': float(t), 'p': float(p)}
    return agg, stats


def finish_experiment(rec, per_obs, n_classes):
    """Aggregate, test, checkpoint and print the section summary."""
    agg, stats = aggregate_and_test(per_obs, n_classes)
    rec.results['aggregate'] = agg
    rec.results['stats'] = stats
    rec.checkpoint('aggregate')
    print(f"[{rec.exp}] n_obs={len(per_obs)}  (target drop @{FRAC_STAT}, mean over obs)")
    for m in ABL_METHODS:
        print(f"  {m:14s} target={np.mean(stats['drops'][m]['target']):+.3f}  "
              f"bystander={np.mean(stats['drops'][m]['bystander']):+.3f}")
    for k, v in stats['tests'].items():
        print(f"  {k}: {v}")
    return agg


def pack_obs(ab, seed, d):
    return {'seed': seed, 'target_class': d, 'baseline': ab.baseline,
            'bft_info': {k: ab.bft_info[k] for k in
                         ('k_star', 'selectivity', 'is_selective', 'warning')},
            'curves': ab.results}


def obs_line(class_names, ab, seed, d):
    tdrop = ab.baseline[d] - ab.results['bft_top'][FRAC_STAT][d]
    print(f'  seed {seed}  class {class_names[d]}: baseline={ab.baseline[d]:.3f}  '
          f'bft_top drop@{FRAC_STAT:.2f}={tdrop:.3f}'
          + ('' if ab.bft_info['is_selective'] else '  [WARN: no selective factor]'))


print(f'MODE={MODE}  DEVICE={DEVICE}  N_TRACE={N_TRACE}  fractions={FRACTIONS}  '
      f'seeds={list(SEED_SUBSET)}  class_cap={CLASS_CAP}')

## §1 · Digit MLP — PUBLICATION SETTINGS

BFT hyperparameters copied verbatim from notebook 02 §1 (the nb09 S10 sweep winner, `rank ×0.7`; see PUBLICATION_SETTINGS.md) — if notebook 02's config changes, update here. Per seed: rebuild the notebook-02 trace (primary mode, test loader, only-correct), then run `ablation_sweep` for every digit against the full MNIST test set. Root branching is one circuit per class (`n_branches=[1, 2, 10]`); inner layers branch twice and the importance path follows each node's first (dominant-factor) branch, as in notebook 13. The root NMF has `k_max=12` but only 10 branches, so a digit whose most-selective factor is one of the two unbranched ones falls back to the first child's path below the root — the root layer itself is always scored with the full selectivity weighting. Checkpoints after every seed.

In [ ]:
from torchvision import datasets
from torchvision.transforms import ToTensor

MLP = {
    'exp': 'mlp_digit',
    'ckpt': 'mnist_digit_mlp_40_20',
    'n_classes': 10,
    'class_names': {i: str(i) for i in range(10)},
    'k_max': [7, 3, 12],            # nb02 §1 K_MAX
    'n_branches': [1, 2, 10],       # nb02 §1 N_BRANCHES
    'stimulus_threshold': 0.7,      # nb02 §1 STIM_THRESHOLD
    'n_random_repeats': 10,
}

# Ablation evaluation set: the full MNIST test set (independent of any trace cap).
mlp_eval_loader = capped_loader(
    datasets.MNIST('../data/', train=False, download=True, transform=ToTensor()))
mlp_classes = target_classes(MLP['n_classes'])

rec_mlp = Recorder(MLP['exp'])
rec_mlp.results['config'] = {
    'ckpt': MLP['ckpt'], 'class_names': MLP['class_names'],
    'k_max': MLP['k_max'], 'n_branches': MLP['n_branches'],
    'stimulus_threshold': MLP['stimulus_threshold'],
    'fractions': FRACTIONS, 'frac_stat': FRAC_STAT, 'methods': ABL_METHODS,
    'n_random_repeats': n_rand(MLP), 'seeds': list(SEED_SUBSET),
    'target_classes': mlp_classes, 'n_trace': N_TRACE, 'bft_max_iter': BFT_MAX_ITER,
    'eval_cap': EVAL_CAP, 'pruning': 'weight-level, all Linear layers',
    'device': str(DEVICE)}
print(f'eval set: {len(mlp_eval_loader.dataset)} samples  targets: {mlp_classes}')

In [ ]:
mlp_obs = []
rec_mlp.results['per_obs'] = mlp_obs

for seed in SEED_SUBSET:
    ed = os.path.join(MODEL_ROOT, f"{MLP['ckpt']}_seed{seed}")
    if not os.path.exists(os.path.join(ed, 'weights.pt')):
        print(f'seed {seed}: no checkpoint under {ed} — skipped '
              f'(train via scripts/train_extra_seeds.sh mlp)')
        continue
    model, config = load_experiment(ed, DEVICE)
    label_transform = get_transform(config['label_transform'])   # None (identity task)
    _, test_loader = get_loaders_from_config(config)
    tl = test_loader if N_TRACE is None else DataLoader(
        Subset(test_loader.dataset, list(range(min(N_TRACE, len(test_loader.dataset))))),
        batch_size=256, shuffle=False)

    coll = collect_layer_dicts(model, tl, label_transform=label_transform, device=DEVICE)
    layer_inputs = [d['input_fmap'] for d in coll['layer_data']]
    layer_names  = [d['name'] for d in coll['layer_data']]
    tree = bft(model, tl, k_max=MLP['k_max'], n_branches=MLP['n_branches'],
               stimulus_threshold=MLP['stimulus_threshold'], weighting='img_selectivity',
               n_jobs=3, **_bft_iter())

    for d in mlp_classes:
        ab = ablation_sweep(model, tree, mlp_eval_loader, target_class=d,
                            fractions=FRACTIONS, methods=ABL_METHODS,
                            label_transform=label_transform, device=DEVICE,
                            n_random_repeats=n_rand(MLP),
                            layer_inputs_list=layer_inputs, layer_names=layer_names,
                            verbose=0)
        mlp_obs.append(pack_obs(ab, seed, d))
        obs_line(MLP['class_names'], ab, seed, d)
    rec_mlp.checkpoint(f'seed{seed}')

print(f'{len(mlp_obs)} observations')

In [ ]:
mlp_agg = finish_experiment(rec_mlp, mlp_obs, MLP['n_classes']) if mlp_obs else None

## §2 · CIFAR CNN — PUBLICATION SETTINGS

BFT hyperparameters verbatim from notebook 03 §1 (the nb09 S10 winner, the `K@R²0.90` rank profile). The trace is layer-dict mode on the confidence-filtered correct test samples (top 60 per class), exactly as in notebook 03 §2, so the traced samples' labels are passed via `targets=` (layer-dict BFTResults carry all-zero targets). Pruning is weight-level over all four conv layers plus the classifier, with conv kernels flattened to `(C_out, C_in·3·3)`. Seeds beyond 0 come from `scripts/train_extra_seeds.sh cnn`; missing checkpoints are skipped, not fatal. `random` uses fewer repeats than on the MLP — it dominates the CNN's evaluation cost, and the mean over 50 observations tightens it anyway.

In [ ]:
import torchvision
import torchvision.transforms as T

CIFAR10_CLASSES = ['airplane', 'automobile', 'bird', 'cat', 'deer',
                   'dog', 'frog', 'horse', 'ship', 'truck']
CNN = {
    'exp': 'cnn_cifar',
    'ckpt': 'cifar10_cnn',
    'n_classes': 10,
    'class_names': {i: c for i, c in enumerate(CIFAR10_CLASSES)},
    'k_max': [4, 3, 4, 6, 10],           # nb03 §1 K_MAX ("K@R²0.90")
    'n_branches': [1, 1, 1, 1, 10],      # nb03 §1 — one root branch per class
    'pool_method': 'avg',                # nb03 §1 POOL_METHOD
    'stimulus_threshold': 0.0,           # nb03 §1 STIM_THRESHOLD
    'n_top_per_class': 60,               # nb03 §2 confidence filter
    'n_random_repeats': 5,
}

CIFAR10_MEAN = (0.4914, 0.4822, 0.4465)
CIFAR10_STD  = (0.2470, 0.2435, 0.2616)
cifar_test = torchvision.datasets.CIFAR10(
    '../data', train=False, download=True,
    transform=T.Compose([T.ToTensor(), T.Normalize(CIFAR10_MEAN, CIFAR10_STD)]))
cnn_eval_loader = capped_loader(cifar_test)
cnn_classes = target_classes(CNN['n_classes'])


def confidence_filter(raw, top_k, n_classes=10):
    """nb03 §2: keep the top_k most confident correct samples per class."""
    keep = np.sort(np.concatenate([
        np.where(raw['targets'] == c)[0][
            np.argsort(raw['confidences'][raw['targets'] == c])[::-1][:top_k]]
        for c in range(n_classes)]))
    layer_data = [{**ld, 'input_fmap': ld['input_fmap'][keep],
                   'output_fmap': ld['output_fmap'][keep]}
                  for ld in raw['layer_data']]
    return {'images': raw['images'][keep], 'targets': raw['targets'][keep],
            'confidences': raw['confidences'][keep], 'layer_data': layer_data}, keep


rec_cnn = Recorder(CNN['exp'])
rec_cnn.results['config'] = {
    'ckpt': CNN['ckpt'], 'class_names': CNN['class_names'],
    'k_max': CNN['k_max'], 'n_branches': CNN['n_branches'],
    'pool_method': CNN['pool_method'],
    'stimulus_threshold': CNN['stimulus_threshold'],
    'n_top_per_class': CNN['n_top_per_class'],
    'fractions': FRACTIONS, 'frac_stat': FRAC_STAT, 'methods': ABL_METHODS,
    'n_random_repeats': n_rand(CNN), 'seeds': list(SEED_SUBSET),
    'target_classes': cnn_classes, 'n_trace': N_TRACE, 'bft_max_iter': BFT_MAX_ITER,
    'eval_cap': EVAL_CAP,
    'pruning': 'weight-level, all Conv2d (flattened kernels) + classifier',
    'device': str(DEVICE)}
print(f'eval set: {len(cnn_eval_loader.dataset)} samples  targets: {cnn_classes}')

In [ ]:
cnn_obs = []
rec_cnn.results['per_obs'] = cnn_obs
RNG = np.random.default_rng(0)

for seed in SEED_SUBSET:
    ed = os.path.join(MODEL_ROOT, f"{CNN['ckpt']}_seed{seed}")
    if not os.path.exists(os.path.join(ed, 'weights.pt')):
        print(f'seed {seed}: no checkpoint under {ed} — skipped '
              f'(train via scripts/train_extra_seeds.sh cnn)')
        continue
    model, _ = load_experiment(ed, DEVICE)

    raw = collect_layer_dicts(model, cnn_eval_loader, DEVICE, only_correct=True)
    data, _ = confidence_filter(raw, CNN['n_top_per_class'])
    del raw
    if N_TRACE is not None and len(data['targets']) > N_TRACE:
        keep = np.sort(RNG.choice(len(data['targets']), N_TRACE, replace=False))
        data['targets'] = data['targets'][keep]
        data['layer_data'] = [dict(ld, input_fmap=ld['input_fmap'][keep],
                                   output_fmap=ld['output_fmap'][keep])
                              for ld in data['layer_data']]
    layer_names  = [ld['name'] for ld in data['layer_data']]
    layer_inputs = [ld['input_fmap'] for ld in data['layer_data']]
    tree = bft(data['layer_data'], k_max=CNN['k_max'], n_branches=CNN['n_branches'],
               conv_pool_method=CNN['pool_method'],
               stimulus_threshold=CNN['stimulus_threshold'],
               weighting='img_selectivity', n_jobs=3, **_bft_iter())
    print(f"seed {seed}: trace on {len(data['targets'])} samples  layers={layer_names}")

    for d in cnn_classes:
        ab = ablation_sweep(model, tree, cnn_eval_loader, target_class=d,
                            fractions=FRACTIONS, methods=ABL_METHODS,
                            label_transform=None, device=DEVICE,
                            n_random_repeats=n_rand(CNN),
                            layer_inputs_list=layer_inputs, layer_names=layer_names,
                            targets=data['targets'], verbose=0)
        cnn_obs.append(pack_obs(ab, seed, d))
        obs_line(CNN['class_names'], ab, seed, d)
    rec_cnn.checkpoint(f'seed{seed}')
    del data, tree

print(f'{len(cnn_obs)} observations')

In [ ]:
cnn_agg = finish_experiment(rec_cnn, cnn_obs, CNN['n_classes']) if cnn_obs else None

## §3 · ImageNet CNN — PUBLICATION SETTINGS

BFT hyperparameters verbatim from notebook 05 §1 (the nb10 winner, `rank ×1.3`); the trace mirrors notebook 05 §2: squeeze-spine layer dicts (initial conv, the 8 fire-module squeeze convs, classifier conv) on the correctly-classified focus-val samples, top 100 per category by confidence. There is a single pretrained model, so the observations are the 8 categories (the Wilcoxon specificity test can reach p ≈ 0.008 at n = 8).

Three ImageNet-specific pieces: (1) the prunable pool is exactly the traced spine — for **all** rankings, so the baselines compete on the same weights; (2) accuracy is per super-category: predictions (argmax over 1000 logits) and labels are both mapped through the category table, and non-focus predictions count as wrong; (3) the Taylor loss is computed on the raw 1000-class labels of the category's samples (`taylor_on_raw_labels=True`) because the output space is finer than the task classes. The root traces 5 branches over 10 factors (nb05's `n_branches`), so categories whose most-selective factor has no branch reuse the first child's path below the root — the root layer itself is always scored with the full selectivity weighting.

Needs ImageNet val images under `../data` (torchvision `ImageNet` root or an `ImageFolder` at `../data/val`); skipped gracefully when absent (e.g. locally).

In [ ]:
IMN = {
    'exp': 'imagenet_cnn',
    'n_classes': 8,
    'k_max': [5, 3, 2, 2, 3, 2, 5, 5, 5, 10],       # nb05 §1 K_MAX ("rank ×1.3")
    'n_branches': [1, 1, 1, 1, 1, 1, 1, 1, 2, 5],   # nb05 §1 N_BRANCHES
    'pool_method': 'avg',                            # nb05 §1 POOL_METHOD
    'stimulus_threshold': 0.0,                       # nb05 §1 STIM_THRESHOLD
    'n_per_category': 100,                           # nb05 §1 N_SAMPLES_PER_CATEGORY
    'n_random_repeats': 10,
}

CATEGORY_CLASSES = {
    'airplane': [404, 895], 'ship': [403, 724], 'car': [609, 751],
    'bicycle': [444, 671], 'elephant': [101, 385], 'bear': [294, 297],
    'dog': [151, 251], 'bird': [7, 9]}
CATEGORY_NAMES = list(CATEGORY_CLASSES.keys())
IDX_TO_CAT = {idx: ci for ci, idxs in enumerate(CATEGORY_CLASSES.values()) for idx in idxs}
IMN['class_names'] = {i: n for i, n in enumerate(CATEGORY_NAMES)}
imn_classes = target_classes(IMN['n_classes'])

# Labels AND argmax-over-1000 predictions are mapped into category space; a
# prediction outside the 16 focus classes maps to -1 and always counts as wrong.
cat_transform = lambda t: torch.as_tensor([IDX_TO_CAT.get(int(x), -1) for x in t])

IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD  = (0.229, 0.224, 0.225)
imagenet_tf = T.Compose([T.Resize(256), T.CenterCrop(224), T.ToTensor(),
                         T.Normalize(IMAGENET_MEAN, IMAGENET_STD)])

try:
    try:
        imn_ds = torchvision.datasets.ImageNet('../data', split='val',
                                               transform=imagenet_tf)
        imn_targets = np.array(imn_ds.targets)
    except Exception:
        imn_ds = torchvision.datasets.ImageFolder('../data/val', transform=imagenet_tf)
        imn_targets = np.array([t for _, t in imn_ds.samples])
    focus_idx = np.where(np.isin(imn_targets, sorted(IDX_TO_CAT)))[0]
    focus_loader = DataLoader(Subset(imn_ds, focus_idx), batch_size=64,
                              shuffle=False, num_workers=4)
    imagenet_available = True
    print(f'focus val samples: {len(focus_idx)} ({len(CATEGORY_NAMES)} categories)')
except Exception as e:
    imagenet_available = False
    print(f'ImageNet val data not found — §3 will be skipped ({e})')


def filter_by_category(raw, n_per_category):
    """nb05 §2: map ImageNet indices → categories, keep top-n by confidence each."""
    orig = raw['targets']
    cats = np.array([IDX_TO_CAT.get(int(t), -1) for t in orig])
    keep = []
    for ci in range(len(CATEGORY_NAMES)):
        ci_idx = np.where(cats == ci)[0]
        ci_idx = ci_idx[np.argsort(raw['confidences'][ci_idx])[::-1]][:n_per_category]
        keep.append(ci_idx)
    keep = np.sort(np.concatenate(keep))
    layer_data = [{**ld, 'input_fmap': ld['input_fmap'][keep],
                   'output_fmap': ld['output_fmap'][keep]}
                  for ld in raw['layer_data']]
    return {'targets': cats[keep], 'orig_targets': orig[keep],
            'layer_data': layer_data}, keep

In [ ]:
imn_obs = []
if imagenet_available:
    from torchvision.models import squeezenet1_1, SqueezeNet1_1_Weights

    rec_imn = Recorder(IMN['exp'])
    rec_imn.results['config'] = {
        'ckpt': 'squeezenet1_1 (torchvision pretrained)',
        'class_names': IMN['class_names'], 'category_classes': CATEGORY_CLASSES,
        'k_max': IMN['k_max'], 'n_branches': IMN['n_branches'],
        'pool_method': IMN['pool_method'],
        'stimulus_threshold': IMN['stimulus_threshold'],
        'n_per_category': IMN['n_per_category'],
        'fractions': FRACTIONS, 'frac_stat': FRAC_STAT, 'methods': ABL_METHODS,
        'n_random_repeats': n_rand(IMN), 'seeds': [0],
        'target_classes': imn_classes, 'n_trace': N_TRACE,
        'bft_max_iter': BFT_MAX_ITER, 'eval_cap': EVAL_CAP,
        'pruning': 'weight-level, traced squeeze spine only (all rankings share the pool)',
        'device': str(DEVICE)}
    rec_imn.results['per_obs'] = imn_obs

    model = squeezenet1_1(weights=SqueezeNet1_1_Weights.IMAGENET1K_V1).to(DEVICE).eval()

    def squeezenet_spine_filter(name, mod):
        """nb05 §2: initial conv, fire-module squeeze convs, classifier conv."""
        return (name in ('features.0', 'classifier.1') or
                (isinstance(mod, nn.Conv2d) and name.endswith('.squeeze')))

    print('collecting spine layer dicts …')
    raw = collect_layer_dicts(model, focus_loader, DEVICE, only_correct=True,
                              layer_filter=squeezenet_spine_filter)
    data, _ = filter_by_category(raw, IMN['n_per_category'])
    del raw
    layer_names  = [ld['name'] for ld in data['layer_data']]
    layer_inputs = [ld['input_fmap'] for ld in data['layer_data']]
    tree = bft(data['layer_data'], k_max=IMN['k_max'], n_branches=IMN['n_branches'],
               conv_pool_method=IMN['pool_method'],
               stimulus_threshold=IMN['stimulus_threshold'],
               weighting='img_selectivity', n_jobs=3, **_bft_iter())
    print(f"trace on {len(data['targets'])} samples  layers={layer_names}")

    for ci in imn_classes:
        ab = ablation_sweep(model, tree, focus_loader, target_class=ci,
                            fractions=FRACTIONS, methods=ABL_METHODS,
                            label_transform=cat_transform, device=DEVICE,
                            n_random_repeats=n_rand(IMN),
                            layer_inputs_list=layer_inputs, layer_names=layer_names,
                            targets=data['targets'], pred_transform=cat_transform,
                            taylor_on_raw_labels=True, verbose=0)
        imn_obs.append(pack_obs(ab, 0, ci))
        obs_line(IMN['class_names'], ab, 0, ci)
        rec_imn.checkpoint(f'cat{ci}')
    print(f'{len(imn_obs)} observations')
else:
    print('§3 skipped — no ImageNet val data')

In [ ]:
imn_agg = finish_experiment(rec_imn, imn_obs, IMN['n_classes']) if imn_obs else None

## §4 · Preview figure

Quick look at all three results (the paper version gets restyled from the JSONs): per model, target-class accuracy vs pruned fraction per ranking (left) and bystander accuracy (right). Specific circuit pruning should show `bft_top` dropping the target class fastest while leaving the bystander classes comparatively intact.

In [ ]:
COLORS = {'bft_top': '#e15759', 'bft_bottom': '#f28e2b', 'magnitude': '#76b7b2',
          'act_magnitude': '#59a14f', 'taylor': '#af7aa1', 'random': '#333333'}
LABELS = {'bft_top': 'BFT most important', 'bft_bottom': 'BFT least important',
          'magnitude': 'Magnitude', 'act_magnitude': 'Act. magnitude',
          'taylor': 'Taylor', 'random': 'Random'}
PANELS = [(t, a) for t, a in [('digit MLP', mlp_agg), ('CIFAR CNN', cnn_agg),
                              ('ImageNet CNN', imn_agg)] if a is not None]

fig, axes = plt.subplots(len(PANELS), 2, figsize=(9, 3.2 * len(PANELS)),
                         sharey=True, squeeze=False)
for r, (title, agg) in enumerate(PANELS):
    fx = np.array(agg['fractions'])
    for ax, key, sub in [(axes[r, 0], 'target', 'target class'),
                         (axes[r, 1], 'bystander', 'bystander classes')]:
        for m in ABL_METHODS:
            mu = np.asarray(agg['methods'][m][f'{key}_mean'])
            sd = np.asarray(agg['methods'][m][f'{key}_sd'])
            ax.plot(fx, mu, '-o', ms=3, lw=1.4, color=COLORS[m], label=LABELS[m])
            ax.fill_between(fx, mu - sd, mu + sd, color=COLORS[m], alpha=0.15, lw=0)
        ax.set(title=f'{title} — {sub} ({agg["n_obs"]} obs)', ylim=(0, 1.05))
    axes[r, 0].set_ylabel('accuracy')
for ax in axes[-1]:
    ax.set_xlabel('fraction of weights pruned')
axes[0, 1].legend(fontsize=7, loc='lower left')
fig.suptitle('Class-circuit pruning — mean ± sd over (seed × class) observations',
             y=1.005)
fig.tight_layout()
p = os.path.join(FIG_DIR, 'fig_pruning_preview.png')
fig.savefig(p, bbox_inches='tight')
print('saved', os.path.relpath(p, REPO))
plt.show()

## How to run on the cluster

`NB14_MODE=cluster` removes the laptop caps (full trace sets, bft's default 500 NMF iters, all 5 seeds, all 10 classes / 8 categories, the full 10-point fraction grid, full eval sets, full random repeats). A GPU is strongly recommended — each observation evaluates the full test set ~100–150 times.

```bash
cd notebooks
NB14_MODE=cluster \
  ../.venv/bin/python -m nbconvert --to notebook --execute \
  --output executed_14_pruning_digit_cnn_imagenet.ipynb \
  --ExecutePreprocessor.timeout=200000 14_pruning_digit_cnn_imagenet.ipynb
```

Needs:
- `mnist_digit_mlp_40_20_seed{0..4}` under `data/models/` (`scripts/train_extra_seeds.sh mlp`, CPU minutes)
- `cifar10_cnn_seed{0..4}` under `data/models/` (`scripts/train_extra_seeds.sh cnn`, GPU, ~150 epochs each)
- ImageNet val images under `data/` (torchvision `ImageNet` root or an `ImageFolder` at `data/val`); MNIST and CIFAR-10 download automatically

Missing CNN seeds and missing ImageNet data are skipped with a message, not fatal; every seed (ImageNet: every category) checkpoints, so a killed run keeps whatever finished. Rough GPU budget: digit MLP well under an hour, CIFAR CNN 1–3 h (50 observations), ImageNet 30–60 min.

Output: `data/results/nb14_pruning_{mlp_digit,cnn_cifar,imagenet_cnn}.json` — same per-experiment schema as nb13's JSON. Copy them into `logs/results/` alongside the nb09/nb13 outputs and plot from `results['aggregate']` (fractions include the 0-point baseline; the JSON round-trip turns the raw `per_obs` curve keys into strings — the aggregate arrays are the ones to draw from).